1. 인코더 삭제
2. 포지셔널 인코딩 삭제
3. 포지셔널 임베딩 추가
4. 디코더 블록 12개로 변경
5. 활성화 함수 변경

In [1]:
%pwd

'c:\\Users\\ADMIN\\Documents\\projects\\AIFFEL_quest_eng\\Main_Quest\\Quest02'

In [2]:
import os
import random
import math
import re
import logging

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from tqdm.autonotebook import tqdm

# NLP 관련: Mecab을 통한 1차 정규화 필수
from konlpy.tag import Mecab 
import sentencepiece as spm

# Evaluation: 실무 표준 지표 활용
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6368\684199018.py:12: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [3]:
from pathlib import Path

In [4]:
class PathManager:
    """환경 및 모듈 간 규격 클래스"""
    """상황에 맞게 경로와 규격을 수정 후 사용할 것"""

    def __init__(self):
        try:
            self.current_working_dir = Path(__file__).resolve().parent
        except NameError:
            self.current_working_dir = Path.cwd()

        self.data_dir = self.current_working_dir / "data"
        self.data_dir.mkdir(parents = True, exist_ok = True)

        self.font_dir = self.current_working_dir / "Fonts"
        self.font_dir.mkdir(parents = True, exist_ok = True)

    def get_current_working_dir(self) -> Path:
        return self.current_working_dir

    def get_data_dir(self) -> Path:
        return self.data_dir

    def get_font_path(self, font_name: str = "NanumGothic.ttf") -> Path:
        return self.font_dir / font_name

In [5]:
class FileManager:

    def __init__(self, file_path: Path):
        self.file_path = file_path
        self.file_name_only = file_path.stem
        self.parent_dir = file_path.parent
        self.extract_dir = self.parent_dir / self.file_name_only
    
    def url_download(self, url: str) -> None:
        if self.file_path.exists():
            print("이미 같은 이름의 파일이 존재합니다.")
            return

        try:
            print("다운로드 중..")
            urllib.request.urlretrieve(url, self.file_path)
            print("다운로드 완료")

        except Exception as e:
            print(f"다운로드 실패, {e}")
            if self.file_path.exists():
                self.file_path.unlink()
                print("불완전한 파일 삭제, 다시 실행해주세요.")

    def unzip(self) -> None:
        self.extract_dir = self.parent_dir / self.file_name_only
        if self.extract_dir.exists():
            print("이미 압축해제된 파일이 존재합니다.")
            return

        print("압축해제 중..")
        with zipfile.ZipFile(self.file_path, "r") as zip_ref:
            top_level = zip_ref.namelist()[0].split('/')[0]
                        
            if top_level == self.file_name_only:
                zip_ref.extractall(self.parent_dir)
            else:
                zip_ref.extractall(self.extract_dir)
            print("압축해제 완료")

    def get_data_path(self, data_name) -> Path:
        return self.extract_dir / data_name

In [6]:
paths = PathManager()
cwd = paths.get_current_working_dir()

data_dir = paths.get_data_dir()

In [ ]:
# import json
# import glob
# import re
# import os
# from tqdm import tqdm

# def clean_text(text):
#     """법률 데이터 내 특수 태그 및 무의미한 공백 제거"""
#     if not text:
#         return ""
#     # 1. <이미지>, <삭제> 등 불필요한 태그 제거
#     text = re.sub(r'<[^>]+>', '', text)
#     # 2. 연속된 공백 및 줄바꿈을 하나의 공백으로 통합
#     text = re.sub(r'\s+', ' ', text).strip()
#     return text

# def build_legal_corpus(input_pattern, output_path, separator="<EOT>"):
#     # 파일 리스트 확보
#     file_list = glob.glob(input_pattern)
#     print(f"📦 총 {len(file_list)}개의 파일을 처리합니다.")

#     success_count = 0
#     with open(output_path, 'w', encoding='utf-8') as f_out:
#         for filename in tqdm(file_list, desc="Generating Corpus"):
#             try:
#                 with open(filename, 'r', encoding='utf-8') as f_in:
#                     data = json.load(f_in)
                    
#                     # 'taskinfo' 내부의 데이터 추출
#                     task_info = data.get('taskinfo', {})
#                     sentences = task_info.get('sentences', [])
#                     q_input = task_info.get('input', "")
#                     a_output = task_info.get('output', "")
                    
#                     # 본문, 질문, 답변을 모두 하나로 병합
#                     combined_text = " ".join([clean_text(s) for s in sentences if s.strip()])
#                     combined_text += " " + clean_text(q_input)
#                     combined_text += " " + clean_text(a_output)
                    
#                     final_text = combined_text.strip()
                    
#                     if final_text:
#                         # 문서의 끝에 <EOT>를 붙여 저장 (한 줄에 한 문서)
#                         f_out.write(f"{final_text} {separator}\n")
#                         success_count += 1
                        
#             except (json.JSONDecodeError, IOError, KeyError):
#                 # 에러 발생 시 해당 파일만 스킵하여 파이프라인 유지
#                 continue

#     print(f"\n✅ 전처리 완료: {success_count}개 문서 기록됨.")
#     if os.path.exists(output_path):
#         size_mb = os.path.getsize(output_path) / (1024 * 1024)
#         print(f"📊 최종 파일 크기: {size_mb:.2f} MB")

# # 사용 시 input_pattern에 *.json을 포함한 경로를 넣으세요.
# build_legal_corpus('./data/laws/*.json', './data/legal_corpus_73k.txt')

📦 총 73065개의 파일을 처리합니다.


Generating Corpus: 100%|██████████| 73065/73065 [00:47<00:00, 1545.70it/s]


✅ 전처리 완료: 73065개 문서 기록됨.
📊 최종 파일 크기: 887.76 MB


In [ ]:
# from mecab import MeCab  # python3-mecab-ko 전용 import
# from tqdm import tqdm
# import os

# def mecab_pre_tokenize(input_file, output_file):
#     # 인스턴스 생성 (Tagger 호출 없이 바로 생성)
#     mecab = MeCab()
    
#     # 출력 경로 폴더가 없다면 생성
#     os.makedirs(os.path.dirname(output_file), exist_ok=True)
    
#     with open(input_file, 'r', encoding='utf-8') as f_in, \
#          open(output_file, 'w', encoding='utf-8') as f_out:
        
#         for line in tqdm(f_in, desc="Mecab Pre-tokenizing"):
#             line = line.strip()
#             if not line:
#                 continue
            
#             # 1. <EOT> 토큰 보호: 텍스트 본문과 구분자 분리
#             if "<EOT>" in line:
#                 # <EOT>를 제외한 본문만 추출
#                 content = line.replace("<EOT>", "").strip()
#                 # 2. 형태소 분석 후 공백으로 연결
#                 morphs = " ".join(mecab.morphs(content))
#                 # 3. 뒤에 <EOT> 다시 결합 (학습 시 시그널 보존)
#                 final_line = f"{morphs} <EOT>\n"
#             else:
#                 # <EOT>가 없는 일반 라인 처리
#                 final_line = " ".join(mecab.morphs(line)) + "\n"
            
#             f_out.write(final_line)

# # 실행 (현재 경로 및 파일명 확인 필수)
# mecab_pre_tokenize('./data/legal_corpus_73k.txt', './data/law_corpus_morphs.txt')

In [ ]:
import sentencepiece as spm

def train_legal_tokenizer(input_file, model_prefix, vocab_size=16000):
    # 300MB 코퍼스 규모를 고려하여 16,000 ~ 24,000 사이의 vocab_size 추천
    sp_args = [
        f"--input={input_file}",                # MeCab으로 처리된 말뭉치 파일
        f"--model_prefix={model_prefix}",       # 저장될 모델 이름
        f"--vocab_size={vocab_size}",           # 어휘 사전 크기
        "--model_type=bpe",                     # GPT-1 표준 알고리즘
        "--max_sentence_length=9999",           # 판결문은 문장이 길 수 있으므로 넉넉히 설정
        "--character_coverage=1.0",             # 한국어/한자 모두 커버하기 위해 1.0 권장
        "--pad_id=0",                           # 패딩 토큰 (보통 0)
        "--unk_id=1",                           # 알 수 없는 토큰
        "--bos_id=-1",                          # GPT-1은 SOS/BOS를 명시적으로 쓰지 않음
        "--eos_id=-1",                          # 대신 우리가 만든 <EOT>를 사용함
        "--user_defined_symbols=<EOT>"          # 가장 중요: MeCab에서 보존한 <EOT>를 특수 토큰으로 지정
    ]
    
    spm.SentencePieceTrainer.train(" ".join(sp_args))
    print(f"✨ 토크나이저 학습 완료: {model_prefix}.model, {model_prefix}.vocab 생성")

# 실행: MeCab 결과물을 넣어 학습시킵니다.
train_legal_tokenizer('./data/law_corpus_morphs.txt', 'legal_gpt_spm', vocab_size=16000)

✨ 토크나이저 학습 완료: legal_gpt_spm.model, legal_gpt_spm.vocab 생성


In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm

class LegalDataset(Dataset):
    def __init__(self, file_path, spm_model_path, max_seq_len=256):
        self.max_seq_len = max_seq_len
        
        # 1. 토크나이저 로드
        self.sp = spm.SentencePieceProcessor()
        self.sp.load(spm_model_path)
        
        # 2. 데이터 로드 (메모리 효율을 위해 제너레이터 방식을 권장하나, 300MB는 리스트로 처리 가능)
        with open(file_path, 'r', encoding='utf-8') as f:
            self.lines = [line.strip() for line in f if line.strip()]

    def __len__(self):
        return len(self.lines)

    def __getitem__(self, idx):
        line = self.lines[idx]
        
        # 3. 토큰화 및 인덱스 변환
        # SentencePiece가 이미 Mecab 결과물을 기반으로 학습되었으므로 바로 인코딩
        tokens = self.sp.encode_as_ids(line)
        
        # 4. 길이 조정 (Trimming or Padding)
        # GPT-1은 학습 시 시퀀스 길이를 맞춰줘야 함
        if len(tokens) > self.max_seq_len:
            tokens = tokens[:self.max_seq_len]
        else:
            # 패딩 토큰(0)으로 채움
            tokens = tokens + [0] * (self.max_seq_len - len(tokens))
            
        # 5. x(입력)와 y(타겟) 생성
        # y는 x를 왼쪽으로 한 칸 쉬프트한 결과 (다음 토큰 예측)
        x = torch.tensor(tokens[:-1], dtype=torch.long)
        y = torch.tensor(tokens[1:], dtype=torch.long)
        
        return x, y

# 사용 예시
dataset = LegalDataset('./data/law_corpus_morphs.txt', 'legal_gpt_spm.model', max_seq_len=256)

In [2]:
def get_legal_dataloader(dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True):
    return DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=shuffle, 
        num_workers=num_workers, # 전달받은 인자를 사용
        pin_memory=pin_memory
    )

In [ ]:
import torch
import torch.nn as nn

class GPT1Embedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len, dropout=0.1):
        super().__init__()
        # 1. Token Embedding: 단어 번호를 벡터로 변환
        self.token_emb = nn.Embedding(vocab_size, d_model)
        
        # 2. Positional Embedding: 위치 번호를 벡터로 변환 (학습 가능)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(self, x):
        # x shape: [batch_size, seq_len]
        seq_len = x.size(1)
        
        # 위치 인덱스 생성 (0, 1, 2, ..., seq_len-1)
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0) # [1, seq_len]
        
        # 토큰 임베딩 + 위치 임베딩
        # GPT-1은 이 두 벡터를 더해서(Element-wise sum) 입력값으로 사용함
        out = self.token_emb(x) + self.pos_emb(positions)
        
        return self.dropout(out) # [batch_size, seq_len, d_model]

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.fc = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size = x.size(0)
        
        # Q, K, V 생성 및 Head 분할
        q = self.w_q(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = self.w_k(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = self.w_v(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # Scaled Dot-Product Attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.d_k ** 0.5)
        
        if mask is not None:
            # scores의 데이터 타입에 맞춰 안전한 최소값을 동적으로 선택합니다.
            fill_value = torch.finfo(scores.dtype).min if scores.dtype == torch.float16 else -1e9
            scores = scores.masked_fill(mask == 0, fill_value)
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        return self.fc(out)

In [5]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        # GPT-1은 보통 d_ff = 4 * d_model을 사용함
        self.linear1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.activation = nn.GELU() # GPT-1의 핵심 선택

    def forward(self, x):
        return self.linear2(self.dropout(self.activation(self.linear1(x))))

In [6]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # 1. Sub-layer: Masked Self-Attention + Residual + LayerNorm
        attn_out = self.attn(x, mask)
        x = self.ln1(x + self.dropout(attn_out))
        
        # 2. Sub-layer: Feed-Forward + Residual + LayerNorm
        ff_out = self.ff(x)
        x = self.ln2(x + self.dropout(ff_out))
        
        return x

In [7]:
import torch
import torch.nn as nn

class GPT1(nn.Module):
    def __init__(self, vocab_size, d_model=768, n_layers=12, n_heads=12, d_ff=3072, max_seq_len=256, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        
        # 1. Embedding Layer
        self.embedding = GPT1Embedding(vocab_size, d_model, max_seq_len, dropout)
        
        # 2. Transformer Blocks (12x)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        
        # 3. Final Language Modeling Head (Text Prediction)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # 가중치 공유 (Weight Tying): Token Embedding과 Output Projection 가중치를 공유하여 파라미터 절약
        self.lm_head.weight = self.embedding.token_emb.weight
        
        # 4. Weight Initialization
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)

    def generate_mask(self, seq_len, device):
        # 미래 토큰을 가리는 상삼각 행렬 마스크 (Causal Mask)
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device)).view(1, 1, seq_len, seq_len)
        return mask

    def forward(self, x):
        batch_size, seq_len = x.size()
        
        # Causal Mask 생성
        mask = self.generate_mask(seq_len, x.device)
        
        # Forward pass
        x = self.embedding(x)
        
        for block in self.blocks:
            x = block(x, mask)
            
        # LM Head를 통해 로짓(Logits) 계산
        logits = self.lm_head(x) # [batch, seq_len, vocab_size]
        
        return logits

In [8]:
import torch.nn as nn

# ignore_index=0을 통해 패딩 토큰이 손실값에 기여하지 않도록 설정
# 이는 모델이 아무 의미 없는 여백을 학습하느라 에너지를 낭비하는 것을 방지함
criterion = nn.CrossEntropyLoss(ignore_index=0)

In [9]:
import torch
from torch.optim import AdamW
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler

def train(model, dataloader, epochs, device, lr=2.5e-4):
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scaler = GradScaler() 
    
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            
            # 1. Forward Pass with Mixed Precision
            with autocast():
                logits = model(x)
                loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            
            # 2. Backward Pass with Scaling
            scaler.scale(loss).backward()
            
            # 3. Unscale for Gradient Clipping (정석적인 순서)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # 4. Step and Update
            scaler.step(optimizer)
            scaler.update()
            
            # 5. 로깅용 데이터 축적
            total_loss += loss.item()
            pbar.set_postfix(loss=loss.item())
            
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")

In [ ]:
# 1. 토크나이저 로드 및 Vocab Size 확인
import sentencepiece as spm
sp = spm.SentencePieceProcessor()
sp.load('legal_gpt_spm.model')
vocab_size = sp.get_piece_size()

# 2. 모델 인스턴스 생성 (설계도대로 집 짓기)
# 하이퍼파라미터는 300MB 데이터 규모에 맞춰 조정 가능합니다.
model = GPT1(
    vocab_size=vocab_size, 
    d_model=256, 
    n_layers=12, 
    n_heads=8, 
    max_seq_len=256
)

# 3. 데이터셋 및 데이터로더 객체 생성
dataset = LegalDataset('./data/law_corpus_morphs.txt', 'legal_gpt_spm.model', max_seq_len=256)
# dataloader = get_legal_dataloader(dataset, batch_size=16)
# DataLoader 생성 부분 수정
dataloader = get_legal_dataloader(
    dataset, 
    batch_size=4, 
    shuffle=True, 
    num_workers=0,  # 0으로 수정하여 데드락 방지
    pin_memory=True
)

# 4. 디바이스 설정
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 5. 이제 훈련 루프 실행
train(model, dataloader, epochs=10, device=device)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19500\530580481.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch 1/10:   0%|          | 0/10333 [00:00<?, ?it/s]C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19500\530580481.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1/10: 100%|██████████| 10333/10333 [08:24<00:00, 20.46it/s, loss=1.27] 


Epoch 1 Average Loss: 2.1702


Epoch 2/10: 100%|██████████| 10333/10333 [08:27<00:00, 20.35it/s, loss=1.72] 


Epoch 2 Average Loss: 1.6689


Epoch 3/10: 100%|██████████| 10333/10333 [08:33<00:00, 20.12it/s, loss=1.26] 


...

Epoch 8 Average Loss: 1.3353


Epoch 9/10: 100%|██████████| 10333/10333 [08:28<00:00, 20.33it/s, loss=1.55] 


Epoch 9 Average Loss: 1.3153


Epoch 10/10: 100%|██████████| 10333/10333 [08:28<00:00, 20.31it/s, loss=1.15] 

Epoch 10 Average Loss: 1.2983


In [11]:
import torch

print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}")
print(f"현재 사용 중인 GPU 장치: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA 사용 가능 여부: True
현재 사용 중인 GPU 장치: NVIDIA GeForce RTX 4070 SUPER


In [25]:
import torch
import torch.nn.functional as F

def generate_legal_text(model, sp, prompt, max_gen_len=100, device='cuda', temperature=0.7, top_p=0.9):
    model.to(device)
    model.eval() 
    
    # 1. 프롬프트 토큰화
    input_ids = sp.encode_as_ids(prompt)
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)
    
    # <EOT> 토큰 ID 확인
    eot_id = sp.piece_to_id('<EOT>')
    
    print(f"Prompt: {prompt}")
    print("-" * 30)
    
    generated = list(input_ids) # 텐서 조작을 위해 리스트로 관리
    
    with torch.no_grad():
        for _ in range(max_gen_len):
            # 현재까지의 입력을 모델에 피딩
            logits = model(input_tensor)[:, -1, :]
            
            # 2. Temperature scaling (확률 분포 조정)
            logits = logits / temperature
            
            # 3. Top-p (Nucleus) Sampling 로직
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            
            # 누적 확률 임계값을 넘는 토큰들 식별
            sorted_indices_to_remove = cumulative_probs > top_p
            # 첫 번째 토큰은 무조건 남기도록 쉬프트
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0
            
            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            logits[:, indices_to_remove] = -float('Inf')
            
            # 4. 확률 기반 샘플링
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
            
            generated.append(next_token)
            
            # 다음 이터레이션을 위한 텐서 업데이트 (Window size 256 준수)
            curr_input = generated[-256:] # max_seq_len 고려
            input_tensor = torch.tensor([curr_input], dtype=torch.long).to(device)
            
            if next_token == eot_id:
                break

    return sp.decode_ids(generated)

# 실행: 이제 temperature와 top_p가 함수 내부에서 인식됩니다.
result = generate_legal_text(model, sp, "원고에게 손해배상", device=device)
print(f"\nGenerated: {result}")

Prompt: 원고에게 손해배상
------------------------------

Generated: 원고에게 손해배상 청구 ( 피고 는 원고 의 위 와 같 은 불법행위 를 원인 으로 한 손해 배상 청구 를 하 고 있 다 ) 는 제 1 심 에서 의 주장 과 크 게 다르 지 않 고 , 제 1 심 에서 제출 된 증거 에다가 이 법원 에 제출 된 각 증거 를 보태 어 보 더라도 제 1 심의 사실 인 정과 판단 은 정당 한 것 으로 인정 된다 . ○ 제 1 심 공동 피 고 A ( 이하 ‘ A ’ 라 한다 ) 는 원고 와 A 의 부모 이 고 , 원고


In [26]:
print(model)

GPT1(
  (embedding): GPT1Embedding(
    (token_emb): Embedding(16000, 256)
    (pos_emb): Embedding(256, 256)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (attn): MultiHeadAttention(
        (w_q): Linear(in_features=256, out_features=256, bias=True)
        (w_k): Linear(in_features=256, out_features=256, bias=True)
        (w_v): Linear(in_features=256, out_features=256, bias=True)
        (fc): Linear(in_features=256, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ff): FeedForward(
        (linear1): Linear(in_features=256, out_features=3072, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=3072, out_features=256, bias=True)
        (activation): GELU(approximate='none')
      )
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (drop